# S6E10 | All-Categorical CatBoost Variants (candidate members)

In the meta-stack, the two all-categorical CatBoosts of `s6e10-ablation-members-oof` carry more weight than any
other new member (one of them has the 5th-largest absolute weight of all 244). This notebook trains five more
variants of that model on the GPU and saves their out-of-fold and test predictions. The meta-stack notebook screens
each one on top of the full stack and keeps only those that help.

The features, folds and original-data model are exactly those of the Encodings + CatBoost + RealMLP notebook
(`StratifiedKFold(5, shuffle=True, random_state=42)`).

**Settings:** GPU T4 x2; attach the competition data and the original survey (the same inputs as that notebook).
Run with `SMOKE = True` first (a few minutes), then `SMOKE = False` (roughly two hours).

In [ ]:
import gc, glob, os, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
from catboost import CatBoostClassifier
from catboost.utils import get_gpu_device_count
warnings.filterwarnings('ignore')

SEED, TARGET, N_SPLITS = 42, 'satisfaction', 5
SMOKE = True      # True: 20k training rows and few trees, to check the notebook end to end
GPU = get_gpu_device_count() > 0
OUT = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
print(f'GPU: {GPU} | smoke run: {SMOKE}')

## 1. Data, preprocessing and features

Unchanged from the Encodings + CatBoost + RealMLP notebook: the original-data model (`orig_pred`), its per-value
rates (`orig_te`), value counts (`freq`), and the fold-safe target encodings (`te`, `te_pair`) built once per fold.

In [ ]:
def find(name, hint):
    hits = sorted(p for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if hint in p)
    assert hits, f'{name} ({hint}) not found under /kaggle/input'
    return hits[0]


KAGGLE = os.path.exists('/kaggle/input')
train_df = pd.read_csv(find('train.csv', 'playground-series-s6e10') if KAGGLE else '../data/train.csv')
test_df = pd.read_csv(find('test.csv', 'playground-series-s6e10') if KAGGLE else '../data/test.csv')
FEATURES = [c for c in test_df.columns if c != 'id']
CAT_COLS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
CONT_COLS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [c for c in FEATURES if c not in CAT_COLS + CONT_COLS]       # the 13 service ratings, 0-5
NUM_COLS = CONT_COLS + RATINGS


def load_original():
    """The original survey: any attached CSV with the competition's columns and a satisfaction label.
    Works with the 129,880-row single file and with the train/test pair of the original Kaggle dataset."""
    paths = (glob.glob('/kaggle/input/**/*.csv', recursive=True) if KAGGLE else ['../data/original_data.csv'])
    frames = []
    for path in sorted(p for p in paths if 'playground-series-s6e10' not in p):
        cols = pd.read_csv(path, nrows=0).columns
        if TARGET in cols and set(FEATURES) <= set(cols):
            frames.append(pd.read_csv(path, usecols=FEATURES + [TARGET]))
    assert frames, 'attach the original Airline Passenger Satisfaction dataset as an input'
    orig = pd.concat(frames, ignore_index=True).drop_duplicates(ignore_index=True)
    orig[TARGET] = orig[TARGET].astype(str).str.strip().str.lower().isin(['true', 'satisfied', '1']).astype('int8')
    return orig


orig_df = load_original()
train_df[TARGET] = train_df[TARGET].astype(str).str.strip().str.lower().eq('true').astype('int8')
if SMOKE:
    train_df = train_df.sample(20_000, random_state=SEED).reset_index(drop=True)
    test_df = test_df.head(5_000).copy()
y, y_orig = train_df[TARGET].values, orig_df[TARGET].values
print(f'train {train_df.shape} | test {test_df.shape} | original {orig_df.shape} | '
      f'positive rate {y.mean():.4f} (original {y_orig.mean():.4f})')

In [ ]:
CAT_MAPS = {
    'Gender': {'Female': 0, 'Male': 1},
    'Customer Type': {'disloyal Customer': 0, 'Loyal Customer': 1},
    'Type of Travel': {'Personal Travel': 0, 'Business travel': 1},
    'Class': {'Eco': 0, 'Eco Plus': 1, 'Business': 2},
}


def encode(df):
    out = df[FEATURES].copy()
    for c, m in CAT_MAPS.items():
        unknown = set(out[c].dropna().unique()) - set(m)
        assert not unknown, f'{c}: unmapped levels {unknown}'
        out[c] = out[c].map(m).astype('float32')
    return out


train, test, orig = encode(train_df), encode(test_df), encode(orig_df)
print(f'encoded train {train.shape}, test {test.shape}, original {orig.shape} | '
      f'NaN left for the trees: {int(train.isna().sum().sum())} train, {int(test.isna().sum().sum())} test')
display(train.head(3))

In [ ]:
PRIOR_ORIG = y_orig.mean()
FOLDS = list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(train, y))


def smoothed_rate(keys, target, prior, smooth=20):
    g = pd.DataFrame({'k': np.asarray(keys), 't': target}).groupby('k')['t'].agg(['sum', 'count'])
    return (g['sum'] + smooth * prior) / (g['count'] + smooth)


# --- orig_pred: LightGBM on the original survey (5-fold average) ---
p_orig = dict(objective='binary', metric='auc', n_estimators=5000, learning_rate=0.03, num_leaves=63,
              min_child_samples=20, colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
              random_state=SEED, n_jobs=-1, verbose=-1)
orig_pred_train, orig_pred_test, orig_oof = np.zeros(len(train)), np.zeros(len(test)), np.zeros(len(orig))
for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(orig, y_orig):
    m = lgb.LGBMClassifier(**p_orig).fit(orig.iloc[tr], y_orig[tr], eval_set=[(orig.iloc[va], y_orig[va])],
                                         callbacks=[lgb.early_stopping(200, verbose=False)])
    orig_oof[va] = m.predict_proba(orig.iloc[va])[:, 1]
    orig_pred_train += m.predict_proba(train)[:, 1] / 5
    orig_pred_test += m.predict_proba(test)[:, 1] / 5
print(f'original-data model: AUC {roc_auc_score(y_orig, orig_oof):.4f} on its own labels, '
      f'{roc_auc_score(y, orig_pred_train):.4f} on the competition labels  <- the label-noise ceiling')

# --- static blocks: orig_pred, orig_te, freq ---
static = {}
for name, df in (('train', train), ('test', test)):
    block = {'orig_pred': orig_pred_train if name == 'train' else orig_pred_test}
    for c in FEATURES:
        block[f'orig_te_{c}'] = df[c].map(smoothed_rate(orig[c], y_orig, PRIOR_ORIG)).to_numpy()
    for c in CONT_COLS:
        block[f'freq_{c}'] = df[c].map(pd.concat([train[c], test[c]]).value_counts()).to_numpy(dtype='float64')
    static[name] = pd.concat([df, pd.DataFrame(block, index=df.index)], axis=1)

# --- te / te_pair keys: one integer per value combination, shared by train and test ---
TE_SPECS = [('Age',), ('Flight Distance',), ('Class', 'Flight Distance'), ('Type of Travel', 'Flight Distance'),
            ('Class', 'Age'), ('Type of Travel', 'Age'), ('Customer Type', 'Age')]
both = pd.concat([train, test], ignore_index=True)
TE_KEYS = {}
for cols in TE_SPECS:
    code_ = np.zeros(len(both), dtype=np.int64)
    for c in cols:
        k = pd.factorize(both[c], use_na_sentinel=False)[0]
        code_ = code_ * (k.max() + 1) + k
    TE_KEYS[cols] = (code_[:len(train)], code_[len(train):])
del both


def fold_te(tr, va):
    """te / te_pair for one fold: (training rows, validation rows, test rows), no row sees its own label."""
    prior, out = y[tr].mean(), ({}, {}, {})
    inner = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, y[tr]))
    for cols in TE_SPECS:
        name = 'te_' + '_x_'.join(c.lower().replace(' ', '_') for c in cols)
        k_train, k_test = TE_KEYS[cols]
        k_tr, col = pd.Series(k_train[tr]), np.full(len(tr), np.nan)
        for a, b in inner:
            col[b] = k_tr.iloc[b].map(smoothed_rate(k_tr.iloc[a], y[tr][a], prior)).to_numpy()
        rate_ = smoothed_rate(k_tr, y[tr], prior)
        out[0][name] = col
        out[1][name] = pd.Series(k_train[va]).map(rate_).to_numpy()
        out[2][name] = pd.Series(k_test).map(rate_).to_numpy()
    return out


# Every fold's matrices are built once and shared by the four tree models
FOLD_DATA = []
for tr, va in FOLDS:
    t_tr, t_va, t_te = fold_te(tr, va)
    FOLD_DATA.append(tuple(pd.concat([base, pd.DataFrame(t, index=base.index)], axis=1) for base, t in
                           ((static['train'].iloc[tr], t_tr), (static['train'].iloc[va], t_va), (static['test'], t_te))))
print('features per model:', FOLD_DATA[0][0].shape[1])

## 2. Variants

Every variant adds a string copy of every raw column as a CatBoost categorical feature, so CatBoost builds its own
ordered target statistics and their combinations. They differ in depth, regularisation, combination depth, whether
our own `te_*` encodings stay next to CatBoost's statistics, and the seed:

| variant | depth | `l2_leaf_reg` | lr | `max_ctr_complexity` | own `te_*` | inputs | seed |
|---|---|---|---|---|---|---|---|
| `d8_l10` | 8 | 10 | 0.05 | 2 | dropped | all features | 42 |
| `d6_l10_c3` | 6 | 10 | 0.06 | 3 | dropped | all features | 42 |
| `d7_l30_te` | 7 | 30 | 0.06 | 2 | kept | all features | 42 |
| `d6_l3_te_s7` | 6 | 3 | 0.06 | 2 | kept | all features | 7 |
| `d6_l10_raw` | 6 | 10 | 0.06 | 2 | — | raw columns only | 42 |

The existing members are `cat_allcat_v1` (depth 6, `l2` 3, own TE kept, CPU) and `cat_allcat_v2` (depth 6, `l2` 10,
own TE dropped, CPU); these GPU runs also differ from them in how CatBoost computes its statistics on the GPU.
Each variant is saved as soon as it finishes, so a re-run skips the ones already done.

In [ ]:
%%time
N_EST = 300 if SMOKE else 20_000
ES = 50 if SMOKE else 300
CB_DEVICE = dict(task_type='GPU') if GPU else dict(thread_count=-1)
TWINS = [f'{c}_cat' for c in FEATURES]

VARIANTS = {
    'd8_l10':      dict(depth=8, l2_leaf_reg=10.0, learning_rate=0.05, max_ctr_complexity=2, keep_te=False, raw=False),
    'd6_l10_c3':   dict(depth=6, l2_leaf_reg=10.0, learning_rate=0.06, max_ctr_complexity=3, keep_te=False, raw=False),
    'd7_l30_te':   dict(depth=7, l2_leaf_reg=30.0, learning_rate=0.06, max_ctr_complexity=2, keep_te=True, raw=False),
    'd6_l3_te_s7': dict(depth=6, l2_leaf_reg=3.0, learning_rate=0.06, max_ctr_complexity=2, keep_te=True, raw=False, seed=7),
    'd6_l10_raw':  dict(depth=6, l2_leaf_reg=10.0, learning_rate=0.06, max_ctr_complexity=2, keep_te=False, raw=True),
}


def prepare(X, v):
    """The variant's inputs plus a string twin of every raw column."""
    if v['raw']:
        X = X[FEATURES]
    elif not v['keep_te']:
        X = X.drop(columns=[c for c in X.columns if c.startswith('te_')])
    twins = {f'{c}_cat': X[c].fillna(-1).astype('int64').astype(str) for c in FEATURES}
    return pd.concat([X, pd.DataFrame(twins, index=X.index)], axis=1)


for tag, v in VARIANTS.items():
    name = f'allcat_{tag}'
    if os.path.exists(f'{OUT}/oof_{name}.npy') and not SMOKE:
        print(f'{name}: already done'); continue
    t0, oof, pred = time.time(), np.zeros(len(y)), np.zeros(len(test))
    try:
        for k, (tr, va) in enumerate(FOLDS):
            X_tr, X_va, X_te = (prepare(X, v) for X in FOLD_DATA[k])
            m = CatBoostClassifier(iterations=N_EST, learning_rate=v['learning_rate'], depth=v['depth'],
                                   l2_leaf_reg=v['l2_leaf_reg'], max_ctr_complexity=v['max_ctr_complexity'],
                                   eval_metric='Logloss', random_seed=v.get('seed', SEED), verbose=0,
                                   allow_writing_files=False, **CB_DEVICE)
            m.fit(X_tr, y[tr], eval_set=(X_va, y[va]), cat_features=TWINS, early_stopping_rounds=ES,
                  use_best_model=True)
            oof[va] = m.predict_proba(X_va)[:, 1]
            pred += m.predict_proba(X_te)[:, 1] / N_SPLITS
            print(f'  {name} fold {k}: {roc_auc_score(y[va], oof[va]):.6f}  iter {m.get_best_iteration()}')
            del m, X_tr, X_va, X_te; gc.collect()
        print(f'{name:22s} OOF AUC {roc_auc_score(y, oof):.6f}   ({(time.time() - t0) / 60:.1f} min)')
        if not SMOKE:
            np.save(f'{OUT}/oof_{name}.npy', oof)
            np.save(f'{OUT}/test_{name}.npy', pred)
    except Exception as e:                   # one failing variant must not cost the others
        print(f'{name} failed: {type(e).__name__}: {str(e)[:200]}')

# member card for the meta-stack: every variant whose files exist
done = sorted(f[4:-4] for f in os.listdir(OUT) if f.startswith('oof_allcat_') and f.endswith('.npy'))
card = pd.DataFrame({'member': done, 'oof_auc': [roc_auc_score(y, np.load(f'{OUT}/oof_{m}.npy')) for m in done]})
if not SMOKE:
    card.to_csv(f'{OUT}/members.csv', index=False)
display(card.round(6))